# Usage
- select the job_gpu_preemtable
- gpu: A100
- instance size: medium


# Camera–LiDAR fusion model on nuScenes

## Goal

This notebook documents the setup of a camera–LiDAR fusion 3D object detector on nuScenes using MMDetection3D.

The objective is to train a BEVFusion model that can be directly compared against the LiDAR-only baseline and used to test whether combining camera and LiDAR information improves detection performance.

## Experimental setting

The fusion model is trained on:
- the nuScenes dataset prepared in the previous notebook
- the same reproducible 20% training-scene subset used for the baseline
- the standard validation split

## Why this fusion model matters

A camera–LiDAR fusion model is important because:
- it tests whether complementary sensor information improves 3D detection
- it allows direct comparison with the LiDAR-only baseline
- it is motivated by the EDA, which suggests that small objects, pedestrian-related classes, and distant objects are likely to benefit from fusion
- BEVFusion provides a unified bird’s-eye-view representation that integrates LiDAR geometry and camera semantics

## Planned outcome

This notebook should:
1. verify the MMDetection3D environment for fusion experiments
2. identify the BEVFusion config to use
3. confirm the subset annotation file paths
4. create a fusion training config for the reduced dataset
5. document the training command
6. prepare the experiment for reproducible execution

In [1]:
# Core libraries
import pandas as pd
import shutil

from pathlib import Path
from typing import List, Tuple, Final
import matplotlib.pyplot as plt

# OpenMMLab stack
import mmengine
import mmdet
import mmdet3d

In [2]:
# Notebook display configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

# Setup and environment

## Define project paths

This notebook uses the same project and dataset paths as the dataset-preparation notebook.

They are defined explicitly here so that the notebook is self-contained and can be run independently.

In [3]:
# Project paths
# PROJECT_ROOT: Final[Path] = Path("/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning")
PROJECT_ROOT = Path.cwd().parent
MMDET3D_ROOT: Final[Path] = PROJECT_ROOT / "external" / "mmdetection3d"

print("PROJECT_ROOT :", PROJECT_ROOT)
print("MMDET3D_ROOT :", MMDET3D_ROOT)

PROJECT_ROOT : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning
MMDET3D_ROOT : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d


## Verify software environment

Before configuring the fusion model, I verify that:
- MMDetection3D is accessible
- the correct Python environment is active
- core libraries are properly installed

This ensures that training will run without unexpected environment issues.

In [4]:
print("mmengine version:", mmengine.__version__)
print("mmdet version   :", mmdet.__version__)
print("mmdet3d version :", mmdet3d.__version__)

mmengine version: 0.10.7
mmdet version   : 3.2.0
mmdet3d version : 1.4.0


The MMDetection3D environment is correctly set up.

Library versions:
- mmengine: 0.10.7
- mmdet: 3.2.0
- mmdet3d: 1.4.0

These versions are compatible and confirm that the training pipeline can be executed.

## Select the model

For the fusion experiment, I use the **BEVFusion** detector, which combines LiDAR point clouds with multi-view camera images in a unified bird’s-eye-view representation for 3D object detection on nuScenes.

**Why BEVFusion**
- explicitly designed for camera–LiDAR fusion on nuScenes
- officially provided in the current MMDetection3D installation
- uses a BEV representation that naturally aligns LiDAR geometry and camera semantics
- enables a direct and fair comparison with the LiDAR-only baseline

**Configuration choice**

I use the official MMDetection3D BEVFusion config:

`bevfusion_lidar-cam_voxel0075_second_secfpn_8xb4-cyclic-20e_nus-3d.py`

This config defines:
- LiDAR-based feature extraction from point clouds
- multi-view camera feature extraction
- fusion in bird’s-eye-view (BEV) space
- a detection head operating on fused BEV features

# Experiment definition

The default training schedule runs for 20 epochs with periodic validation during training. In practice, this results in a runtime of up to 11 days on a single H100 GPU.

To make experimentation feasible, I use a reduced 20% subset of the dataset and adopt a lighter training schedule to:
- reduce wall-clock training time
- allow multiple experiments (baseline, fusion, ablations)
- still obtain meaningful results

This configuration is suitable for:
- debugging the pipeline
- obtaining a first BEVFusion result quickly
- enabling controlled comparisons with the LiDAR-only baseline

## Define the goal of this experiment

The goal of this experiment is to train a **BEVFusion camera–LiDAR model** on nuScenes and evaluate whether combining modalities improves detection performance compared to the LiDAR-only baseline, particularly in challenging scenarios such as small or distant objects.

In [5]:
EXPERIMENT_NAME: Final[str] = "bevfusion_nuscenes_20pct"

print("EXPERIMENT_NAME:", EXPERIMENT_NAME)

EXPERIMENT_NAME: bevfusion_nuscenes_20pct


## Select the model

I use **BEVFusion** as the fusion detector. It combines LiDAR point clouds with multi-view camera images in a unified bird’s-eye-view representation for 3D object detection on nuScenes.

**Rationale**

- explicitly designed for camera–LiDAR fusion on nuScenes
- supported in the current MMDetection3D installation
- enables direct comparison with the LiDAR-only baseline

In [6]:
MODEL_NAME: Final[str] = "BEVFusion"
MODEL_CONFIG_NAME: Final[str] = "bevfusion_lidar-cam_voxel0075_second_secfpn_8xb4-cyclic-20e_nus-3d.py"

print("MODEL_NAME       :", MODEL_NAME)
print("MODEL_CONFIG_NAME:", MODEL_CONFIG_NAME)

MODEL_NAME       : BEVFusion
MODEL_CONFIG_NAME: bevfusion_lidar-cam_voxel0075_second_secfpn_8xb4-cyclic-20e_nus-3d.py


## Define the configuration path

I create a separate experiment config instead of modifying the original MMDetection3D config directly. This keeps the repository clean and makes the experiment easier to reproduce.



In [7]:
SOURCE_CONFIG_PATH: Path = (
    MMDET3D_ROOT
    / "projects"
    / "BEVFusion"
    / "configs"
    / MODEL_CONFIG_NAME
)

EXPERIMENT_CONFIG_DIR: Path = MMDET3D_ROOT / "configs" / "my_experiments"
EXPERIMENT_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_CONFIG_PATH: Path = EXPERIMENT_CONFIG_DIR / "bevfusion_nuscenes_20pct.py"

print("SOURCE_CONFIG_PATH     :", SOURCE_CONFIG_PATH)
print("SOURCE exists          :", SOURCE_CONFIG_PATH.exists())
print("EXPERIMENT_CONFIG_DIR  :", EXPERIMENT_CONFIG_DIR)
print("EXPERIMENT_CONFIG_PATH :", EXPERIMENT_CONFIG_PATH)

SOURCE_CONFIG_PATH     : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/projects/BEVFusion/configs/bevfusion_lidar-cam_voxel0075_second_secfpn_8xb4-cyclic-20e_nus-3d.py
SOURCE exists          : True
EXPERIMENT_CONFIG_DIR  : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/configs/my_experiments
EXPERIMENT_CONFIG_PATH : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/configs/my_experiments/bevfusion_nuscenes_20pct.py


## Copy the fusion config into the experiment folder

I now create a project-specific copy of the BEVFusion config.

This copied config will be the one modified for the reduced dataset experiment.
The original MMDetection3D config remains untouched.

In [9]:
if EXPERIMENT_CONFIG_PATH.exists():
    print("EXPERIMENT_CONFIG_PATH:", EXPERIMENT_CONFIG_PATH)
    print("\nSafety rule: existing file is not overwritten automatically.")
else:
    shutil.copy2(SOURCE_CONFIG_PATH, EXPERIMENT_CONFIG_PATH)
    print("Copied BEVFusion config to:")
    print(EXPERIMENT_CONFIG_PATH)

print("\nExperiment config exists:", EXPERIMENT_CONFIG_PATH.exists())

Copied BEVFusion config to:
/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/configs/my_experiments/bevfusion_nuscenes_20pct.py

Experiment config exists: True


# Config file settings

In this section, I prepare and modify the MMDetection3D config file for the fusion experiment.  
All key settings — dataset, modalities, training schedule, and data loading — are defined through this config.

The main changes define a reduced-budget training setup: a reproducible 20% training subset, a shorter training schedule, and adjusted data-loading settings. These changes reduce compute cost while keeping the experiment suitable for analysis.

The configuration is based on an official BEVFusion nuScenes setup and is adapted to match the reduced dataset and training constraints while remaining consistent with the LiDAR-only baseline.

## Open the config file

In [10]:
with open(EXPERIMENT_CONFIG_PATH, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    print(f"{i+1:03d}: {line.rstrip()}")

001: _base_ = [
002:     './bevfusion_lidar_voxel0075_second_secfpn_8xb4-cyclic-20e_nus-3d.py'
003: ]
004: point_cloud_range = [-54.0, -54.0, -5.0, 54.0, 54.0, 3.0]
005: input_modality = dict(use_lidar=True, use_camera=True)
006: backend_args = None
007: 
008: model = dict(
009:     type='BEVFusion',
010:     data_preprocessor=dict(
011:         type='Det3DDataPreprocessor',
012:         mean=[123.675, 116.28, 103.53],
013:         std=[58.395, 57.12, 57.375],
014:         bgr_to_rgb=False),
015:     img_backbone=dict(
016:         type='mmdet.SwinTransformer',
017:         embed_dims=96,
018:         depths=[2, 2, 6, 2],
019:         num_heads=[3, 6, 12, 24],
020:         window_size=7,
021:         mlp_ratio=4,
022:         qkv_bias=True,
023:         qk_scale=None,
024:         drop_rate=0.0,
025:         attn_drop_rate=0.0,
026:         drop_path_rate=0.2,
027:         patch_norm=True,
028:         out_indices=[1, 2, 3],
029:         with_cp=False,
030:         convert_weights=True

## Define override values

I define the main settings that will override the source config for this experiment. These values describe the reduced-budget training setup used in this notebook.

They include adjustments to the training subset, training schedule, and data-loading parameters, ensuring consistency with the LiDAR-only baseline while preserving the official BEVFusion camera–LiDAR setup.

In [11]:
TRAIN_ANN_FILE: Final[str] = "subsets/nuscenes_infos_train_20pct.pkl"

MAX_EPOCHS: Final[int] = 10
VAL_INTERVAL: Final[int] = 1
TRAIN_CFG: Final[str] = f"dict(by_epoch=True, max_epochs={MAX_EPOCHS}, val_interval={VAL_INTERVAL})"

# BEVFusion is more memory-intensive than LiDAR-only models
TRAIN_BATCH_SIZE: Final[int] = 4
TRAIN_NUM_WORKERS: Final[int] = 16

print("Override values:")
print("TRAIN_ANN_FILE    :", TRAIN_ANN_FILE)
print("MAX_EPOCHS        :", MAX_EPOCHS)
print("VAL_INTERVAL      :", VAL_INTERVAL)
print("TRAIN_CFG         :", TRAIN_CFG)
print("TRAIN_BATCH_SIZE  :", TRAIN_BATCH_SIZE)
print("TRAIN_NUM_WORKERS :", TRAIN_NUM_WORKERS)

Override values:
TRAIN_ANN_FILE    : subsets/nuscenes_infos_train_20pct.pkl
MAX_EPOCHS        : 10
VAL_INTERVAL      : 1
TRAIN_CFG         : dict(by_epoch=True, max_epochs=10, val_interval=1)
TRAIN_BATCH_SIZE  : 4
TRAIN_NUM_WORKERS : 16


## Write a clear and self-contained experiment config

I define an explicit experiment config that includes all required dataset, modality, and pipeline components, while controlling the key experiment settings through a small set of variables. This includes the reduced training subset, the shorter training schedule, the data-loading parameters, and a checkpoint configuration that saves progress regularly during training.

The configuration is based on an official BEVFusion nuScenes setup and is adapted only where necessary to match the reduced dataset and training constraints.

**Rationale**

- keeps the configuration fully explicit and easy to understand  
- makes the experiment reproducible and self-contained  
- allows quick adjustments through a small set of variables  
- simplifies reuse of the same setup across different models  
- adds regular checkpoints for safer long HPC runs  

In [12]:
CONFIG_TEXT: str = f"""_base_ = [
    '../../projects/BEVFusion/configs/{MODEL_CONFIG_NAME}'
]

train_dataloader = dict(
    batch_size={TRAIN_BATCH_SIZE},
    num_workers={TRAIN_NUM_WORKERS},
    persistent_workers=True,
    dataset=dict(
        dataset=dict(
            ann_file='{TRAIN_ANN_FILE}'
        )
    )
)

val_dataloader = dict(
    num_workers={TRAIN_NUM_WORKERS},
    persistent_workers=True
)

test_dataloader = dict(
    num_workers={TRAIN_NUM_WORKERS},
    persistent_workers=True
)

train_cfg = {TRAIN_CFG}

param_scheduler = [
    dict(
        type='LinearLR',
        start_factor=0.33333333,
        by_epoch=False,
        begin=0,
        end=500
    ),
    dict(
        type='CosineAnnealingLR',
        begin=0,
        T_max={MAX_EPOCHS},
        end={MAX_EPOCHS},
        by_epoch=True,
        eta_min_ratio=1e-4,
        convert_to_iter_based=True
    ),
    dict(
        type='CosineAnnealingMomentum',
        eta_min=0.85 / 0.95,
        begin=0,
        end={MAX_EPOCHS * 0.4},
        by_epoch=True,
        convert_to_iter_based=True
    ),
    dict(
        type='CosineAnnealingMomentum',
        eta_min=1,
        begin={MAX_EPOCHS * 0.4},
        end={MAX_EPOCHS},
        by_epoch=True,
        convert_to_iter_based=True
    )
]

default_hooks = dict(
    checkpoint=dict(
        type='CheckpointHook',
        interval=1,
        save_last=True,
        max_keep_ckpts=3
    )
)
"""

In [13]:
with open(EXPERIMENT_CONFIG_PATH, "w") as f:
    f.write(CONFIG_TEXT)

print("Experiment config written to:")
print(EXPERIMENT_CONFIG_PATH)

Experiment config written to:
/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/configs/my_experiments/bevfusion_nuscenes_20pct.py


## Verify updated config settings

I verify that the main override settings were correctly written into the config file. Only the relevant lines are displayed to keep the output concise.

This step ensures that the reduced training subset, training schedule, and data-loading parameters have been correctly applied on top of the base BEVFusion configuration.

In [14]:
KEYWORDS = [
    "ann_file",
    "train_cfg",
    "batch_size",
    "num_workers",
    "T_max",          # uses MAX_EPOCHS
]

with open(EXPERIMENT_CONFIG_PATH, "r") as f:
    lines = f.readlines()

print("Updated config lines:\n")

for i, line in enumerate(lines):
    for keyword in KEYWORDS:
        if keyword in line:
            print(f"{i+1:03d}: {line.rstrip()}")
            # also show next line for context (important for nested dicts)
            if i + 1 < len(lines):
                print(f"{i+2:03d}: {lines[i+1].rstrip()}")
            print()
            break

Updated config lines:

006:     batch_size=4,
007:     num_workers=16,

007:     num_workers=16,
008:     persistent_workers=True,

011:             ann_file='subsets/nuscenes_infos_train_20pct.pkl'
012:         )

017:     num_workers=16,
018:     persistent_workers=True

022:     num_workers=16,
023:     persistent_workers=True

026: train_cfg = dict(by_epoch=True, max_epochs=10, val_interval=1)
027: 

039:         T_max=10,
040:         end=10,



# Training command

The BEVFusion model can be trained using the following command:

```bash
cd /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d

conda activate py38_mmdet3d

export PYTHONPATH=$PWD:$PYTHONPATH

python tools/train.py configs/my_experiments/bevfusion_nuscenes_20pct.py
```

## Training Time Estimation (nuScenes)

I estimate the full training cost based on a reduced experiment:

- 20% of the dataset  
- 5 epochs  
- observed runtime ≈ **14 hours**

From this:

- **Time per epoch (20%)** ≈ 2.8 hours  
- Scaling to full dataset (×5):
  - **Time per epoch (100%)** ≈ 14 hours  

Assuming a standard **20-epoch schedule**:

- **Total training time (full nuScenes)** ≈  
  **14h × 20 ≈ 280 hours (~12 days of compute)**

---

## Practical Implication (UBELIX)

On UBELIX using preemptable A100 GPUs:

- jobs use a defined walltime (e.g., 4h in this setup)
- jobs may be **preempted at any time** by higher-priority jobs
- interrupted jobs are **automatically requeued**
- training resumes from the **last checkpoint**

Therefore, training proceeds as a sequence of shorter runs:

running → interrupted (timeout or preemption) → requeued → running again → resume from checkpoint

---

## Conclusion

Full nuScenes training with BEVFusion is computationally expensive:

- ≈ **280 hours of GPU compute**
- typically **2–3 weeks wall-clock time** under preemptable scheduling

# Running the fusion training with SLURM

For long BEVFusion training runs on UBELIX, it is safer to submit the experiment through SLURM instead of launching it from a fragile interactive terminal.

**Why this is useful**

Using SLURM makes the training run independent of the local computer state.

This means:
- the job keeps running even if the laptop sleeps
- the job is not interrupted by SSH disconnection
- resource requests are explicit and reproducible

**Strategy**

I create a small SLURM submission script that:
- requests one GPU
- targets a GPU type compatible with the current BEVFusion build
- sets an appropriate wall-time limit
- activates the correct conda environment
- sets `PYTHONPATH` for the BEVFusion project import
- launches the MMDetection3D training command
- can resume from the latest checkpoint if needed

This is important because BEVFusion training is computationally heavier than the LiDAR-only baseline and may require multiple sessions depending on the available GPU and wall-time limit.

## Defining the SLURM script location

I store the SLURM submission script inside the project repository so that the full training procedure remains reproducible and well documented.

This ensures that:
- the exact execution setup is version-controlled together with the code
- the same script can be reused across different experiments
- running conditions (resources, environment, command) are explicitly defined and easy to reproduce

Keeping the SLURM script within the project structure also simplifies debugging and future extensions of the training pipeline.

In [15]:
WORK_DIR: Final[Path] = MMDET3D_ROOT / "work_dirs" / "bevfusion_nuscenes_20pct"
TRAIN_CONFIG_PATH: Final[Path] = MMDET3D_ROOT / "configs" / "my_experiments" / "bevfusion_nuscenes_20pct.py"

SLURM_DIR: Final[Path] = PROJECT_ROOT / "slurm"
SLURM_RUN_DIR: Final[Path] = SLURM_DIR / "bevfusion_20pct"
SLURM_RUN_DIR.mkdir(parents=True, exist_ok=True)

SLURM_SCRIPT_PATH: Final[Path] = SLURM_RUN_DIR / "train_bevfusion_nuscenes_20pct.slurm"

print("WORK_DIR         :", WORK_DIR)
print("TRAIN_CONFIG_PATH:", TRAIN_CONFIG_PATH)
print("SLURM_RUN_DIR    :", SLURM_RUN_DIR)
print("SLURM_SCRIPT_PATH:", SLURM_SCRIPT_PATH)

WORK_DIR         : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/work_dirs/bevfusion_nuscenes_20pct
TRAIN_CONFIG_PATH: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/configs/my_experiments/bevfusion_nuscenes_20pct.py
SLURM_RUN_DIR    : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/bevfusion_20pct
SLURM_SCRIPT_PATH: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/bevfusion_20pct/train_bevfusion_nuscenes_20pct.slurm


## Creating a resume-ready SLURM submission script

The script below is designed for a single-GPU UBELIX training run with a 12-hour wall-time limit.

UBELIX requires the GPU type to be specified explicitly in the SLURM request.
For this experiment, I request one A100 GPU, which is compatible with the current BEVFusion setup.

The script supports both:
- a fresh training run if no checkpoint is present
- a resumed training run if a previous checkpoint already exists

https://slurm.schedmd.com/sbatch.html#OPT_signal 

https://hpc-unibe-ch.github.io/runjobs/scheduled-jobs/preemption/ 

In [15]:
# TODO: update nb 02 with: source "$(conda info --base)/etc/profile.d/conda.sh"

**Workflow**
job_gpu_preemptable:  Time Limit = 06:00:00 for a100

- submit job
- job enters **pending**
- job starts **running**

- job is interrupted:
  - either it **reaches the requested time limit** (for example, `4h`)
  - or it is **preempted by a higher-priority job**

- Slurm **terminates the job**
  - state: **TIMEOUT** (if the requested walltime is reached)
  - state: **PREEMPTED** (if killed by the scheduler)

- job is **automatically requeued**
- job returns to **pending**
- job runs again when resources are available
- training **resumes from last checkpoint**

**UBELIX documentation links**
* Partitions & QoS (gpu-invest, preemptable jobs):
https://hpc-unibe-ch.github.io/runjobs/partitions/
* Preemption behavior:
https://hpc-unibe-ch.github.io/runjobs/scheduled-jobs/preemption/
* GPU jobs & limits (A100, CPU/memory per GPU):
https://hpc-unibe-ch.github.io/runjobs/scheduled-jobs/gpus/

In [16]:
SLURM_SCRIPT: str = f"""#!/bin/bash
#
# Description:
#   BEVFusion training on nuScenes (20% subset)
#   using A100 (preemptable) on UBELIX.
#
# Usage:
#   sbatch {SLURM_SCRIPT_PATH}
#
# Check job:
#   squeue -u $USER
#
# Logs:
#   tail -f {SLURM_RUN_DIR}/job_<JOBID>.out
#   tail -f {SLURM_RUN_DIR}/job_<JOBID>.err
#
#SBATCH --job-name=bevfusion_20pct
#SBATCH --output={SLURM_RUN_DIR}/job_%j.out
#SBATCH --error={SLURM_RUN_DIR}/job_%j.err

#SBATCH --account=gratis
#SBATCH --partition=gpu-invest
#SBATCH --qos=job_gpu_preemptable

#SBATCH --time=05:50:00
#SBATCH --nodes=1
#SBATCH --ntasks=1

#SBATCH --gres=gpu:a100:1
#SBATCH --cpus-per-task=16
#SBATCH --mem=48G

set -eo pipefail

# Proper conda initialization (CRITICAL)
source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate py38_mmdet3d

cd {MMDET3D_ROOT}
export PYTHONPATH="$PWD:$PYTHONPATH"

echo "========================================"
echo "Job ID: $SLURM_JOB_ID"
echo "Host: $(hostname)"
echo "Start time: $(date)"
echo "========================================"

which python
python --version
nvidia-smi

LAST_CHECKPOINT_FILE="{WORK_DIR}/last_checkpoint"

if [ -f "$LAST_CHECKPOINT_FILE" ]; then
    CHECKPOINT_PATH=$(cat "$LAST_CHECKPOINT_FILE")
    echo "Resuming from $CHECKPOINT_PATH"
    python tools/train.py {TRAIN_CONFIG_PATH} \\
        --resume "$CHECKPOINT_PATH"
else
    echo "Starting fresh training"
    python tools/train.py {TRAIN_CONFIG_PATH}
fi
"""

In [17]:
print("TRAIN_CONFIG_PATH exists :", TRAIN_CONFIG_PATH.exists())
print("WORK_DIR exists         :", WORK_DIR.exists())

last_ckpt_path = WORK_DIR / "last_checkpoint"
epoch_ckpt_path = WORK_DIR / "epoch_5.pth"

print("last_checkpoint exists  :", last_ckpt_path.exists())
print("epoch_5 exists          :", epoch_ckpt_path.exists())

if last_ckpt_path.exists():
    print("last_checkpoint content:")
    print(last_ckpt_path.read_text().strip())

TRAIN_CONFIG_PATH exists : True
WORK_DIR exists         : False
last_checkpoint exists  : False
epoch_5 exists          : False


In [18]:
with open(SLURM_SCRIPT_PATH, "w") as f:
    f.write(SLURM_SCRIPT)

print("Saved SLURM script to:")
print(SLURM_SCRIPT_PATH)

Saved SLURM script to:
/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/bevfusion_20pct/train_bevfusion_nuscenes_20pct.slurm


## Verifying the generated SLURM script

Before submitting the job, I inspect the generated script to confirm that:
- the requested resources match the current BEVFusion training setup
- the requested GPU type is compatible with the current build
- the environment activation is correct
- `PYTHONPATH` is set correctly for the BEVFusion project import
- the training command points to the correct experiment config
- the script checks for an existing checkpoint and resumes automatically when available

In [19]:
with open(SLURM_SCRIPT_PATH, "r") as f:
    slurm_lines = f.readlines()

for i, line in enumerate(slurm_lines):
    print(f"{i+1:02d}: {line.rstrip()}")

01: #!/bin/bash
02: #
03: # Description:
04: #   BEVFusion training on nuScenes (20% subset)
05: #   using A100 (preemptable) on UBELIX.
06: #
07: # Usage:
08: #   sbatch /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/bevfusion_20pct/train_bevfusion_nuscenes_20pct.slurm
09: #
10: # Check job:
11: #   squeue -u $USER
12: #
13: # Logs:
14: #   tail -f /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/bevfusion_20pct/job_<JOBID>.out
15: #   tail -f /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/bevfusion_20pct/job_<JOBID>.err
16: #
17: #SBATCH --job-name=bevfusion_20pct
18: #SBATCH --output=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/bevfusion_20pct/job_%j.out
19: #SBATCH --error=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/bevfusion_20pct/job_%j.err
20: 
21: #SBATCH --account=gratis
22: #SBATCH --partition=gpu-invest
23: #SBATCH --qos=job_gpu_preemptable
24: 
25: #SBATCH --time

## SLURM submission and resume behavior

The training job can be submitted with:

```bash
cd /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning
sbatch slurm/bevfusion_20pct/train_bevfusion_nuscenes_20pct.slurm
```

### Monitoring  
Useful commands after submission:

```bash
squeue -u ae04q066
```
or
```bash
squeue -u $USER
```

check the effective limits:

```bash
sqos
```
```bash
tail -f /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/bevfusion_20pct/job_<JOBID>.out
```

```bash
cat /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/bevfusion_20pct/job_<JOBID>.err
```

Find your job node
```bash
squeue -u $USER
```

SSH into the node
```bash
ssh gnodeXX
```

Check GPU usage
```bash
nvidia-smi
```

Optional live view:
```bash
watch -n 2 nvidia-smi
```

Exit the node
```bash
exit
```

### Resume behavior  
The SLURM script automatically checks whether the following file exists:

```bash
/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/work_dirs/bevfusion_nuscenes_20pct/last_checkpoint
```

This file contains the path to the most recent saved checkpoint.

- if `last_checkpoint` exists, the script reads the checkpoint path stored inside it and resumes training from that checkpoint  
- if `last_checkpoint` does not exist, training starts from scratch  

This makes it possible to continue training across multiple 12-hour SLURM jobs without manually editing the training command each time.

### Cancel job  

```bash
scancel <JOBID>
```

### Benefit  
This execution method is robust to:

- laptop sleep  
- SSH disconnects  
- unstable local network connections  
- wall-time limits requiring multiple job submissions  

In [20]:
# TODO: add the analysis section

# Loss analysis
## Check the experiment folder

In this step, I check the experiment folder stored in `WORK_DIR`. This confirms that the finished run is available and shows the main files created during training.


In [ ]:
print("Experiment folder:")
print(WORK_DIR)
print()

print("Files in WORK_DIR:")
for item in sorted(WORK_DIR.iterdir()):
    if item.is_dir():
        print("[DIR ]", item.name)
    else:
        print("[FILE]", item.name)

## Select the run and define analysis files

In this step, I define the main files used for the analysis. These files contain the logs and scalar metrics from the selected run.

In [ ]:
from pathlib import Path

RUN_NAME = "20260423_160700" # select the experiment: 20260409_173740, 20260410_212957

RUN_DIR: Path = WORK_DIR / f"{RUN_NAME}"

LOG_FILE: Path = RUN_DIR / f"{RUN_NAME}.log"
JSON_LOG_FILE: Path = RUN_DIR / f"{RUN_NAME}.json"
SCALARS_FILE: Path = RUN_DIR / "vis_data" / "scalars.json"

print("RUN_DIR:")
print(RUN_DIR)
print()

print("Analysis files:")
print("LOG_FILE      =", LOG_FILE)
print("JSON_LOG_FILE =", JSON_LOG_FILE)
print("SCALARS_FILE  =", SCALARS_FILE)

## Load scalar records and inspect metrics

In this step, I load the scalar metrics from `scalars.json`. This file is stored as one JSON record per line, so I read it line by line and store the records in a list.

In [ ]:
# list of run directories
run_names: List[str] = [
    "20260423_160700",
    "20260424_082649",
    "20260424_150344",
    "20260505_121853",
    
]

dfs = []

for run_name in run_names:
    run_dir = WORK_DIR / run_name
    scalars_file = run_dir / "vis_data" / "scalars.json"
    
    # Load JSON Lines file → DataFrame
    df = pd.read_json(scalars_file, lines=True)
    
    # add a column to track source
    df["run"] = run_name
    
    dfs.append(df)

# Stack all DataFrames
scalars_df = pd.concat(dfs, ignore_index=True)

print(scalars_df.head())
print(scalars_df.shape)

In [ ]:
for i, col in enumerate(scalars_df.columns):
    print(i, col)

In [ ]:
scalars_df.tail()

## Plot the training loss curve

In this step, I plot the training loss as a function of the training step. This gives a visual overview of how the model learns over time. A decreasing curve indicates that training is progressing correctly.

In [ ]:
import matplotlib.pyplot as plt

# Sort by step to reconstruct the full training timeline
scalars_df = scalars_df.sort_values("step")

plt.figure()

plt.plot(scalars_df["step"], scalars_df["loss"])

plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training Loss Curve")

plt.grid()

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Sort by step to keep correct order
scalars_df = scalars_df.sort_values("step")

# Filter out very large losses
filtered_df = scalars_df[scalars_df["loss"] < 100]

plt.figure()

plt.plot(filtered_df["step"], filtered_df["loss"])

plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training Loss Curve (Zoomed)")

plt.grid()

plt.show()

# Validation metrics analysis 
## Extract available metrics

In this step, I inspect which validation metrics are present in the scalar records.


In [37]:
# All nuScenes validation metric columns
metrics_cols = [

    # ===================== Car =====================
    "NuScenes metric/pred_instances_3d_NuScenes/car_AP_dist_0.5",
    "NuScenes metric/pred_instances_3d_NuScenes/car_AP_dist_1.0",
    "NuScenes metric/pred_instances_3d_NuScenes/car_AP_dist_2.0",
    "NuScenes metric/pred_instances_3d_NuScenes/car_AP_dist_4.0",
    "NuScenes metric/pred_instances_3d_NuScenes/car_trans_err",
    "NuScenes metric/pred_instances_3d_NuScenes/car_scale_err",
    "NuScenes metric/pred_instances_3d_NuScenes/car_orient_err",
    "NuScenes metric/pred_instances_3d_NuScenes/car_vel_err",
    "NuScenes metric/pred_instances_3d_NuScenes/car_attr_err",

    # ===================== Global Errors =====================
    "NuScenes metric/pred_instances_3d_NuScenes/mATE",
    "NuScenes metric/pred_instances_3d_NuScenes/mASE",
    "NuScenes metric/pred_instances_3d_NuScenes/mAOE",
    "NuScenes metric/pred_instances_3d_NuScenes/mAVE",
    "NuScenes metric/pred_instances_3d_NuScenes/mAAE",

    # ===================== Truck =====================
    "NuScenes metric/pred_instances_3d_NuScenes/truck_AP_dist_0.5",
    "NuScenes metric/pred_instances_3d_NuScenes/truck_AP_dist_1.0",
    "NuScenes metric/pred_instances_3d_NuScenes/truck_AP_dist_2.0",
    "NuScenes metric/pred_instances_3d_NuScenes/truck_AP_dist_4.0",
    "NuScenes metric/pred_instances_3d_NuScenes/truck_trans_err",
    "NuScenes metric/pred_instances_3d_NuScenes/truck_scale_err",
    "NuScenes metric/pred_instances_3d_NuScenes/truck_orient_err",
    "NuScenes metric/pred_instances_3d_NuScenes/truck_vel_err",
    "NuScenes metric/pred_instances_3d_NuScenes/truck_attr_err",

    # ===================== Construction Vehicle =====================
    "NuScenes metric/pred_instances_3d_NuScenes/construction_vehicle_AP_dist_0.5",
    "NuScenes metric/pred_instances_3d_NuScenes/construction_vehicle_AP_dist_1.0",
    "NuScenes metric/pred_instances_3d_NuScenes/construction_vehicle_AP_dist_2.0",
    "NuScenes metric/pred_instances_3d_NuScenes/construction_vehicle_AP_dist_4.0",
    "NuScenes metric/pred_instances_3d_NuScenes/construction_vehicle_trans_err",
    "NuScenes metric/pred_instances_3d_NuScenes/construction_vehicle_scale_err",
    "NuScenes metric/pred_instances_3d_NuScenes/construction_vehicle_orient_err",
    "NuScenes metric/pred_instances_3d_NuScenes/construction_vehicle_vel_err",
    "NuScenes metric/pred_instances_3d_NuScenes/construction_vehicle_attr_err",

    # ===================== Bus =====================
    "NuScenes metric/pred_instances_3d_NuScenes/bus_AP_dist_0.5",
    "NuScenes metric/pred_instances_3d_NuScenes/bus_AP_dist_1.0",
    "NuScenes metric/pred_instances_3d_NuScenes/bus_AP_dist_2.0",
    "NuScenes metric/pred_instances_3d_NuScenes/bus_AP_dist_4.0",
    "NuScenes metric/pred_instances_3d_NuScenes/bus_trans_err",
    "NuScenes metric/pred_instances_3d_NuScenes/bus_scale_err",
    "NuScenes metric/pred_instances_3d_NuScenes/bus_orient_err",
    "NuScenes metric/pred_instances_3d_NuScenes/bus_vel_err",
    "NuScenes metric/pred_instances_3d_NuScenes/bus_attr_err",

    # ===================== Trailer =====================
    "NuScenes metric/pred_instances_3d_NuScenes/trailer_AP_dist_0.5",
    "NuScenes metric/pred_instances_3d_NuScenes/trailer_AP_dist_1.0",
    "NuScenes metric/pred_instances_3d_NuScenes/trailer_AP_dist_2.0",
    "NuScenes metric/pred_instances_3d_NuScenes/trailer_AP_dist_4.0",
    "NuScenes metric/pred_instances_3d_NuScenes/trailer_trans_err",
    "NuScenes metric/pred_instances_3d_NuScenes/trailer_scale_err",
    "NuScenes metric/pred_instances_3d_NuScenes/trailer_orient_err",
    "NuScenes metric/pred_instances_3d_NuScenes/trailer_vel_err",
    "NuScenes metric/pred_instances_3d_NuScenes/trailer_attr_err",

    # ===================== Barrier =====================
    "NuScenes metric/pred_instances_3d_NuScenes/barrier_AP_dist_0.5",
    "NuScenes metric/pred_instances_3d_NuScenes/barrier_AP_dist_1.0",
    "NuScenes metric/pred_instances_3d_NuScenes/barrier_AP_dist_2.0",
    "NuScenes metric/pred_instances_3d_NuScenes/barrier_AP_dist_4.0",
    "NuScenes metric/pred_instances_3d_NuScenes/barrier_trans_err",
    "NuScenes metric/pred_instances_3d_NuScenes/barrier_scale_err",
    "NuScenes metric/pred_instances_3d_NuScenes/barrier_orient_err",
    "NuScenes metric/pred_instances_3d_NuScenes/barrier_vel_err",
    "NuScenes metric/pred_instances_3d_NuScenes/barrier_attr_err",

    # ===================== Motorcycle =====================
    "NuScenes metric/pred_instances_3d_NuScenes/motorcycle_AP_dist_0.5",
    "NuScenes metric/pred_instances_3d_NuScenes/motorcycle_AP_dist_1.0",
    "NuScenes metric/pred_instances_3d_NuScenes/motorcycle_AP_dist_2.0",
    "NuScenes metric/pred_instances_3d_NuScenes/motorcycle_AP_dist_4.0",
    "NuScenes metric/pred_instances_3d_NuScenes/motorcycle_trans_err",
    "NuScenes metric/pred_instances_3d_NuScenes/motorcycle_scale_err",
    "NuScenes metric/pred_instances_3d_NuScenes/motorcycle_orient_err",
    "NuScenes metric/pred_instances_3d_NuScenes/motorcycle_vel_err",
    "NuScenes metric/pred_instances_3d_NuScenes/motorcycle_attr_err",

    # ===================== Bicycle =====================
    "NuScenes metric/pred_instances_3d_NuScenes/bicycle_AP_dist_0.5",
    "NuScenes metric/pred_instances_3d_NuScenes/bicycle_AP_dist_1.0",
    "NuScenes metric/pred_instances_3d_NuScenes/bicycle_AP_dist_2.0",
    "NuScenes metric/pred_instances_3d_NuScenes/bicycle_AP_dist_4.0",
    "NuScenes metric/pred_instances_3d_NuScenes/bicycle_trans_err",
    "NuScenes metric/pred_instances_3d_NuScenes/bicycle_scale_err",
    "NuScenes metric/pred_instances_3d_NuScenes/bicycle_orient_err",
    "NuScenes metric/pred_instances_3d_NuScenes/bicycle_vel_err",
    "NuScenes metric/pred_instances_3d_NuScenes/bicycle_attr_err",

    # ===================== Pedestrian =====================
    "NuScenes metric/pred_instances_3d_NuScenes/pedestrian_AP_dist_0.5",
    "NuScenes metric/pred_instances_3d_NuScenes/pedestrian_AP_dist_1.0",
    "NuScenes metric/pred_instances_3d_NuScenes/pedestrian_AP_dist_2.0",
    "NuScenes metric/pred_instances_3d_NuScenes/pedestrian_AP_dist_4.0",
    "NuScenes metric/pred_instances_3d_NuScenes/pedestrian_trans_err",
    "NuScenes metric/pred_instances_3d_NuScenes/pedestrian_scale_err",
    "NuScenes metric/pred_instances_3d_NuScenes/pedestrian_orient_err",
    "NuScenes metric/pred_instances_3d_NuScenes/pedestrian_vel_err",
    "NuScenes metric/pred_instances_3d_NuScenes/pedestrian_attr_err",

    # ===================== Traffic Cone =====================
    "NuScenes metric/pred_instances_3d_NuScenes/traffic_cone_AP_dist_0.5",
    "NuScenes metric/pred_instances_3d_NuScenes/traffic_cone_AP_dist_1.0",
    "NuScenes metric/pred_instances_3d_NuScenes/traffic_cone_AP_dist_2.0",
    "NuScenes metric/pred_instances_3d_NuScenes/traffic_cone_AP_dist_4.0",
    "NuScenes metric/pred_instances_3d_NuScenes/traffic_cone_trans_err",
    "NuScenes metric/pred_instances_3d_NuScenes/traffic_cone_scale_err",
    "NuScenes metric/pred_instances_3d_NuScenes/traffic_cone_orient_err",
    "NuScenes metric/pred_instances_3d_NuScenes/traffic_cone_vel_err",
    "NuScenes metric/pred_instances_3d_NuScenes/traffic_cone_attr_err",

    # ===================== Summary =====================
    "NuScenes metric/pred_instances_3d_NuScenes/NDS",
    "NuScenes metric/pred_instances_3d_NuScenes/mAP",
]

In [ ]:
metrics_df = scalars_df[["step"] + metrics_cols].dropna(
    subset=["NuScenes metric/pred_instances_3d_NuScenes/NDS"]
)

metrics_df.head()

**Observations**: 

I’m not seeing a bug with checkpointing or resume, those are working fine. The missing NDS values happen because of the Slurm time limit.

Each epoch runs: training → save checkpoint → validation. Sometimes the job hits the 6-hour limit after training and saving, but before validation finishes. When that happens, the checkpoint exists, but no NDS is logged.

When I resume, training continues normally and validation completes for a later epoch. That’s why I only get NDS for epochs 1, 3, and 5—epochs 2 and 4 were interrupted during validation.

## Plot of summary metrics

In this step, I plot the NDS (nuScenes Detection Score) over the validation steps. This shows how the model performance improves across epochs. Unlike training loss, this curve reflects actual detection quality on the validation set.



In [ ]:
# Plot both curves
plt.plot(metrics_df["step"], metrics_df["NuScenes metric/pred_instances_3d_NuScenes/NDS"], marker="o", label="NDS")
plt.plot(metrics_df["step"], metrics_df["NuScenes metric/pred_instances_3d_NuScenes/mAP"], marker="o", label="mAP")

plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("NDS vs mAP over Epochs")

plt.legend()
plt.grid()

plt.show()

## Plot of global errors

In [ ]:
# Plot both curves
plt.plot(metrics_df["step"], metrics_df["NuScenes metric/pred_instances_3d_NuScenes/mATE"], marker="o", label="mATE")
plt.plot(metrics_df["step"], metrics_df["NuScenes metric/pred_instances_3d_NuScenes/mASE"], marker="o", label="mASE")
plt.plot(metrics_df["step"], metrics_df["NuScenes metric/pred_instances_3d_NuScenes/mAOE"], marker="o", label="mAOE")
plt.plot(metrics_df["step"], metrics_df["NuScenes metric/pred_instances_3d_NuScenes/mAVE"], marker="o", label="mAVE")
plt.plot(metrics_df["step"], metrics_df["NuScenes metric/pred_instances_3d_NuScenes/mAAE"], marker="o", label="mAAE")

plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("NDS vs mAP over Epochs")

plt.legend()
plt.grid()

plt.show()